# matrixlayout Binder demo

`matrixlayout` builds LaTeX/TikZ layouts for matrix-centered figures and renders them to SVG. This notebook is a capability tour: Gaussian-elimination grids, QR grids, eigen/SVD tables, back-substitution layouts, selectors, decorators, callouts, and render options.

In [ ]:
from matrixlayout import (
    backsubst_svg,
    decorator_bg,
    decorator_box,
    decorator_color,
    render_eig_svg,
    render_ge_svg,
    render_qr_svg,
    sel_col,
    sel_entry,
    sel_row,
    show_svg,
)

RENDER_OPTS = dict(toolchain_name="pdftex_dvisvgm", crop="tight", padding=(2, 2, 2, 2))

## Basic GE grid

A GE grid is a matrix-of-matrices layout. `None` entries leave holes; ordinary nested lists become matrix blocks.

In [ ]:
matrices = [
    [None, [[1, 2], [3, 4]], [[1], [0]]],
    [[[1, 0], [0, 1]], [[2, 1], [0, 3]], None],
]

ge_basic_svg = render_ge_svg(matrices=matrices, **RENDER_OPTS)
show_svg(ge_basic_svg)

## GE labels, highlights, and callouts

Decoration specs attach visual structure to named grid positions: boxes, guide lines, labels, and delimiter callouts.

In [ ]:
decorations = [
    {"grid": (0, 1), "entries": [(0, 0)], "box": True},
    {"grid": (0, 1), "hlines": 1},
    {"grid": (0, 1), "label": r"\mathbf{A}", "side": "right", "angle": -35, "length": 8},
    {"grid": (1, 0), "label": r"\mathbf{I}", "side": "left", "angle": -35, "length": 8},
]

ge_callout_svg = render_ge_svg(
    matrices=matrices,
    decorations=decorations,
    create_medium_nodes=True,
    **RENDER_OPTS,
)
show_svg(ge_callout_svg)

## Selectors and decorators

Selectors target entries by row, column, or coordinate. Decorators transform only the selected TeX entries.

In [ ]:
entry_decorators = [
    {"grid": (0, 0), "entries": [sel_entry(0, 0)], "decorator": decorator_box(color="blue")},
    {"grid": (0, 0), "entries": [sel_entry(1, 1)], "decorator": decorator_color("red")},
    {"grid": (0, 0), "entries": [sel_col(1)], "decorator": decorator_bg("yellow!25")},
]

decorated_svg = render_ge_svg(
    matrices=[[[1, 2], [3, 4]]],
    decorators=entry_decorators,
    **RENDER_OPTS,
)
show_svg(decorated_svg)

## Row and column labels

Label rows and columns make basis-vector, coordinate, and block-structure diagrams easier to read.

In [ ]:
label_svg = render_ge_svg(
    matrices=[[[1, 0, 2], [0, 1, -1]]],
    label_rows=[{"grid": (0, 0), "side": "above", "rows": [["x1", "x2", "b"]]}],
    label_cols=[{"grid": (0, 0), "side": "left", "cols": [["r1"], ["r2"]]}],
    **RENDER_OPTS,
)
show_svg(label_svg)

## QR / Gram-Schmidt grid

QR grids use the same rendering boundary but add QR-oriented labels, known-zero styling, and callout conventions.

In [ ]:
qr_matrices = [
    [None, None, [[1, 2], [3, 4]], [[1, 0], [0, 1]]],
    [None, [[1, 0], [0, 1]], [[1, 2], [3, 4]], [[1, 0], [0, 1]]],
    [[[1, 0], [0, 1]], [[1, 0], [0, 1]], [[1, 2], [3, 4]], None],
]

qr_svg = render_qr_svg(matrices=qr_matrices, array_names=True, **RENDER_OPTS)
show_svg(qr_svg)

## Eigen/SVD table

Eigen/SVD tables render precomputed layout specs. The package formats and places values; it does not compute decompositions.

In [ ]:
svd_spec = {
    "lambda": [4, 1],
    "ma": [1, 1],
    "sigma": [2, 1],
    "evecs": [[[1, 0]], [[0, 1]]],
    "qvecs": [[[1, 0]], [[0, 1]]],
    "uvecs": [[[1, 0, 0]], [[0, 1, 0]], [[0, 0, 1]]],
    "sz": (3, 2),
}

eig_svg = render_eig_svg(svd_spec, case="SVD", **RENDER_OPTS)
show_svg(eig_svg)

## Back-substitution layout

Back-substitution layouts combine a system statement, a cascade of substitutions, and the final solution.

In [ ]:
backsubst_demo_svg = backsubst_svg(
    system_txt=r"$x + 2y = 5,\quad y = 1$",
    cascade_txt=[r"$x + 2(1) = 5$", r"$x = 3$"],
    solution_txt=r"$(x,y) = (3,1)$",
    **RENDER_OPTS,
)
show_svg(backsubst_demo_svg)

## Render options

`toolchain_name`, `crop`, and `padding` are forwarded through the rendering boundary. This notebook uses `pdftex_dvisvgm` because it is compact and reliable in Binder.

In [ ]:
compact_svg = render_ge_svg(
    matrices=[[[1, 2], [3, 4]]],
    toolchain_name="pdftex_dvisvgm",
    crop="tight",
    padding=(0, 0, 0, 0),
)
show_svg(compact_svg)

## Optional: inspect SVG text

The rendered image is the useful output for most notebook work. Inspect raw SVG only when debugging, saving, or comparing renderer output.

In [ ]:
print(backsubst_demo_svg[:500])

## Troubleshooting

If rendering fails, check that the TeX packages and converters are available. The Binder image checks `systeme.sty` during `postBuild`, and the renderers keep failure artifacts in a temporary directory when a toolchain fails.

In [ ]:
import shutil

checks = ["latex", "dvisvgm", "kpsewhich"]
{name: shutil.which(name) for name in checks}